# Prepare SCIN labels and grouped splits

Run the environment notebook first. This notebook preserves private case/image manifests and reproducible splits in your own Drive. It displays aggregates only. SCIN Data Use License applies.


In [ ]:
from pathlib import Path
import os, subprocess, sys
repo=Path('/content/DermReady_')
os.chdir(repo)
subprocess.run(['git','pull','--ff-only'],check=True)
from google.colab import drive
drive.mount('/content/drive')
persistent=Path('/content/drive/MyDrive/DermReady')
raw=Path('/content/dermready_scin_raw')
if not all((raw/name).exists() for name in ['scin_cases.csv','scin_labels.csv','image_inventory.json']):
    subprocess.run([sys.executable,'scripts/audit_scin.py','--raw-dir',str(raw),'--output-dir',str(persistent/'reports')],check=True)


In [ ]:
subprocess.run([sys.executable,'scripts/prepare_scin.py','--config','configs/experiment.yaml','--raw-dir',str(raw),'--processed-dir',str(persistent/'processed'),'--split-dir',str(persistent/'splits'),'--report-dir',str(persistent/'reports')],check=True)
subprocess.run([sys.executable,'-m','pytest','tests/test_data_loader.py','tests/test_splits.py','-q'],check=True)


In [ ]:
import json, hashlib
report=json.loads((persistent/'reports/split_summary.json').read_text())
for name, expected in report['artifact_sha256'].items():
    directory=persistent/'processed' if name in ['cases.jsonl','images.csv'] else persistent/'splits'
    assert hashlib.sha256((directory/name).read_bytes()).hexdigest()==expected, name
print('Saved artifact hashes verified.')
print(json.dumps(report['split_counts'],indent=2))


## Next

Use the saved train split for training and validation for thresholds. Keep test fixed for final evaluation. Do not commit private manifests or dataset files. The preparation script refuses changed source/seed/fraction provenance unless explicitly replaced for a new experiment.
